In [88]:
from ollama import ChatResponse
from ollama import chat

from pydantic import BaseModel
from typing import List, Tuple


class BBox(BaseModel):
   x1 : int
   y1 : int
   x2 : int
   y2 : int
  
# class BBResult(BaseModel):

#   name: dict[str, BBox]

class InnResult(BaseModel):
    inn : str
    bbox: BBox



def run_model(prompt: str, files: List[str]) -> str:
    
    response: ChatResponse = chat(
	model='qwen3-vl:8b-thinking', 
    format=InnResult.model_json_schema(),
	messages=[
    {
        'role': 'user',
        'content': prompt,
        'images': files
    },
    ])

    return InnResult.model_validate_json(response['message']['content']).model_dump()



In [16]:
import cv2

def get_image_size(path_to_image: str) -> Tuple[int, int]:

    return cv2.imread(path_to_image).shape


In [83]:
path_to_image = '/Users/roman/projects/docs_generator/data/Screenshot 2026-05-19 at 13.24.58.png'
path_to_image = '/Users/roman/projects/docs_generator/data/Screenshot 2026-05-19 at 13.26.21.png'

image_size = get_image_size(path_to_image)
print('image size :' , image_size)

# prompt =  f'Я дал тебе документ ИНН физическрго на русском языке. Размеры документа ({image_size[0]},{image_size[1]})  Верни координаты прямоугольников, содержащих  фамилию имя отчество физического лица  , его дату рождения и номер ИНН. Результат верни в виде json. Названия полей дай соответственно fio, bday и  inn.'

prompt =  'Ты эксперт, проверяющий содержание документов. Я дал тебе документ, содержащий Идентификационный номер налогоплательщика физ лица. Верни значение поля  "Идентификационный номер налогоплательщика" Также верни координаты премоугольника,  содержащего даное поле'


bboxes = run_model(prompt, [path_to_image])

image size : (1706, 1190, 3)


In [84]:
bboxes

{'inn': '583508598609', 'bbox': {'x1': 500, 'y1': 300, 'x2': 650, 'y2': 350}}

In [85]:
import cv2
import numpy as np

def paint_bboxes(path_to_image: str, bboxes):
    img = cv2.imread(path_to_image)
    result = img.copy()
   
    bbox = bboxes['bbox']
    
    cv2.rectangle(result, (bbox['x1']+100, bbox['y1']+370), (bbox['x2']+100, bbox['y2']+370), (0, 0, 255), 2)

    cv2.imwrite('bboxes.jpg',result)      

paint_bboxes(path_to_image, bboxes)

In [101]:
import cv2
import numpy as np

def eraze_bboxes(path_to_image: str, bbox):

    img = cv2.imread(path_to_image)

    # 2. Создание черной маски (размером с исходное изображение)
    mask = np.zeros(img.shape[:2], dtype=np.uint8)

    # 3. Рисование белого контура на маске (координаты области, которую нужно стереть)
    cv2.rectangle(mask, (bbox['x1'], bbox['y1']), (bbox['x2'], bbox['y2']), 255, -1)

    # 4. Удаление объекта
    # Третий параметр — радиус окрестности для восстановления пикселей
    result = cv2.inpaint(img, mask, 3, cv2.INPAINT_TELEA)

    # 5. Сохранение результата
    cv2.imwrite('erased.jpg', result)

bbpx = {'x1':400, 'y1':400, 'x2':500, 'y2':600}
eraze_bboxes('/Users/roman/projects/docs_generator/data/Screenshot 2026-05-19 at 13.23.55.png',  bbpx)
